# Capítulo 5 · Tokenización

Un tokenizador BPE completo, entrenado con el texto de los capítulos 1 a 4 del libro. Python estándar y CPU: no hay que instalar nada. Ejecuta las celdas en orden.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio: el mismo programa, por partes, para probarlo con tus propios textos.

## 1. Caracteres, números y bytes

`ord` da el número Unicode de cada carácter y `encode("utf-8")` lo convierte en bytes. Observa cuántos bytes ocupan la eñe, las tildes y el símbolo del euro.

In [1]:
for texto in ["a", "ñ", "é", "€", "año", "La señal llegó"]:
    print(repr(texto), [ord(c) for c in texto], list(texto.encode("utf-8")))

'a' [97] [97]
'ñ' [241] [195, 177]
'é' [233] [195, 169]
'€' [8364] [226, 130, 172]
'año' [97, 241, 111] [97, 195, 177, 111]
'La señal llegó' [76, 97, 32, 115, 101, 241, 97, 108, 32, 108, 108, 101, 103, 243] [76, 97, 32, 115, 101, 195, 177, 97, 108, 32, 108, 108, 101, 103, 195, 179]


## 2. Contar parejas y fusionarlas

`contar_parejas` cuenta cuántas veces aparece cada pareja de símbolos consecutivos. `fusionar` sustituye una pareja por un símbolo nuevo. En Colab, la primera celda descarga el corpus si no está.

In [2]:
"""Capítulo 5: un tokenizador BPE. Solo biblioteca estándar y CPU."""
from collections import Counter
from pathlib import Path

CORPUS = Path("material/corpus_capitulos_1_4.txt")
if not CORPUS.exists():  # en Colab, descargamos el corpus del repositorio
    import urllib.request
    CORPUS.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/corpus_capitulos_1_4.txt", CORPUS)
FUSIONES = 256
FIN = "<|fin|>"


def contar_parejas(ids):
    return Counter(zip(ids, ids[1:]))


def fusionar(ids, pareja, nuevo):
    resultado, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pareja:
            resultado.append(nuevo)
            i += 2
        else:
            resultado.append(ids[i])
            i += 1
    return resultado

## 3. Aprender las fusiones

`entrenar` repite: contar, elegir la pareja más frecuente y fusionarla. Comprueba las cuatro fusiones del ejemplo a mano del capítulo.

In [3]:
def entrenar(texto, fusiones=FUSIONES):
    ids = list(texto.encode("utf-8"))
    reglas = {}
    vocabulario = {i: bytes([i]) for i in range(256)}
    for k in range(fusiones):
        parejas = contar_parejas(ids)
        if not parejas:
            break
        pareja = max(parejas, key=parejas.get)
        if parejas[pareja] < 2:
            break
        nuevo = 256 + k
        ids = fusionar(ids, pareja, nuevo)
        reglas[pareja] = nuevo
        vocabulario[nuevo] = vocabulario[pareja[0]] + vocabulario[pareja[1]]
    return reglas, vocabulario

In [4]:
reglas4, voc4 = entrenar("la casa la masa la pasa", 4)
for pareja, nuevo in reglas4.items():
    print(nuevo, pareja, repr(voc4[nuevo].decode("utf-8")))

256 (97, 32) 'a '
257 (108, 256) 'la '
258 (97, 115) 'as'
259 (258, 256) 'asa '


## 4. Codificar y decodificar

`codificar` aplica las reglas en el orden en que se aprendieron; `decodificar` une los bytes y recupera el texto.

In [5]:
def codificar(texto, reglas):
    ids = list(texto.encode("utf-8"))
    for pareja, nuevo in reglas.items():
        ids = fusionar(ids, pareja, nuevo)
    return ids


def decodificar(ids, vocabulario):
    return b"".join(vocabulario[i] for i in ids).decode("utf-8", errors="replace")


def trozos(ids, vocabulario):
    return [vocabulario[i].decode("utf-8", errors="replace") for i in ids]

In [6]:
for texto in ["la masa", "las casas"]:
    ids = codificar(texto, reglas4)
    print(repr(texto), ids, trozos(ids, voc4), repr(decodificar(ids, voc4)))

reglas_n, voc_n = entrenar("niño niña niños", 3)
print("Token con media eñe:", voc_n[257], "->", repr(voc_n[257].decode("utf-8", errors="replace")))

'la masa' [257, 109, 258, 97] ['la ', 'm', 'as', 'a'] 'la masa'
'las casas' [108, 258, 32, 99, 258, 258] ['l', 'as', ' ', 'c', 'as', 'as'] 'las casas'
Token con media eñe: b'ni\xc3' -> 'ni�'


## 5. Unir documentos con un token especial

In [7]:
def unir_documentos(documentos, reglas, id_fin):
    ids = []
    for documento in documentos:
        ids.extend(codificar(documento, reglas))
        ids.append(id_fin)
    return ids

## 6. El experimento completo

Entrena 256 fusiones con el corpus del libro, codifica el corpus completo y comprueba la ida y vuelta. Tarda unos veinte segundos.

In [8]:
def ejecutar():
    texto = CORPUS.read_text(encoding="utf-8")
    reglas, vocabulario = entrenar(texto)
    id_fin = len(vocabulario)
    vocabulario[id_fin] = FIN.encode("utf-8")
    print("Vocabulario:", len(vocabulario), "tokens")
    nuevos = list(reglas.values())
    print("Primeras fusiones:", trozos(nuevos[:20], vocabulario))
    largos = sorted(nuevos, key=lambda i: len(vocabulario[i]), reverse=True)
    print("Tokens más largos:", trozos(largos[:10], vocabulario))

    ids = codificar(texto, reglas)
    n_bytes = len(texto.encode("utf-8"))
    print("Corpus:", n_bytes, "bytes ->", len(ids), "tokens")
    print("Bytes por token: %.3f" % (n_bytes / len(ids)))
    print("Ida y vuelta exacta:", decodificar(ids, vocabulario) == texto)

    for frase in ("La señal llegó", "probabilidades", "ferrocarril",
                  "La red aprende a contar", "The network learns to count"):
        ids_frase = codificar(frase, reglas)
        print(repr(frase), len(frase.encode("utf-8")), "bytes ->",
              len(ids_frase), "tokens:", trozos(ids_frase, vocabulario))

    documentos = unir_documentos(["ana", "bala"], reglas, id_fin)
    print("Dos documentos:", documentos, repr(decodificar(documentos, vocabulario)))
    print("El texto <|fin|> escrito por alguien:", codificar(FIN, reglas))
    return reglas, vocabulario

In [9]:
reglas, vocabulario = ejecutar()

Vocabulario: 513 tokens
Primeras fusiones: ['a ', 's ', 'e ', 'o ', 'en', 're', 'ar', 'os ', 'n ', 'er', 'ci', 'l ', 'co', '. ', 'ad', 'la ', 'es', 'de ', 'as ', 'un']
Tokens más largos: ['probabilidad', 'capítulo ', 'pérdida ', 'aciones ', 'ación ', 'ciones ', 'probabi', 'capítu', 'amente ', 'aremos ']
Corpus: 122497 bytes -> 52183 tokens
Bytes por token: 2.347
Ida y vuelta exacta: True
'La señal llegó' 16 bytes -> 8 tokens: ['La ', 'se', 'ñ', 'al ', 'l', 'le', 'g', 'ó']
'probabilidades' 14 bytes -> 2 tokens: ['probabilidad', 'es']
'ferrocarril' 11 bytes -> 8 tokens: ['f', 'er', 'r', 'o', 'c', 'ar', 'ri', 'l']
'La red aprende a contar' 23 bytes -> 8 tokens: ['La ', 'red ', 'apr', 'en', 'de ', 'a ', 'cont', 'ar']
'The network learns to count' 27 bytes -> 18 tokens: ['T', 'h', 'e ', 'n', 'e', 't', 'w', 'or', 'k', ' ', 'le', 'ar', 'n', 's ', 'to ', 'co', 'un', 't']
Dos documentos: [292, 97, 512, 98, 284, 97, 512] 'ana<|fin|>bala<|fin|>'
El texto <|fin|> escrito por alguien: [60, 124, 444

## 7. Prueba con tus propios textos

Cambia las frases de la lista y observa cómo las trocea el tokenizador. Prueba palabras largas, nombres propios, números y textos en otros idiomas.

Fíjate en la `¡`: ocupa dos bytes que el tokenizador no ha fusionado. Al mostrar cada token por separado, esos bytes aparecen como el carácter de sustitución, aunque `decodificar` recupera la frase completa sin problemas.

In [10]:
for frase in ["El tokenizador trocea el texto", "1234567", "¡Hola, mundo!"]:
    ids = codificar(frase, reglas)
    print(len(ids), "tokens:", trozos(ids, vocabulario))

18 tokens: ['E', 'l ', 'to', 'k', 'en', 'i', 'z', 'ad', 'or ', 'tr', 'o', 'c', 'e', 'a ', 'el ', 't', 'ex', 'to']
7 tokens: ['1', '2', '3', '4', '5', '6', '7']
11 tokens: ['�', '�', 'H', 'o', 'l', 'a, ', 'm', 'un', 'd', 'o', '!']


## Ejercicios

1. Escribe los bytes UTF-8 de `pingüino`. ¿Cuántos caracteres y cuántos bytes tiene? ¿Por qué no coinciden?

2. ¿Cuántos bytes ocupan `camión` y `CAMION`? Explica la diferencia.

3. Repite a mano las cuatro fusiones de `la casa la masa la pasa` y comprueba las longitudes de la tabla del capítulo.

4. En la segunda fusión había un empate. Repite el ejemplo eligiendo la otra pareja. Después de cuatro fusiones, ¿cambian los trozos en que queda partido el texto? ¿Y los números de los tokens?

5. Codifica `masa la` con las cuatro reglas y decodifica después los identificadores obtenidos.

6. Codifica `salsa` con las cuatro reglas. ¿Qué reglas se aplican? ¿Qué ocurre con las palabras que no estaban en el texto de entrenamiento?

7. En el ejemplo `niño niña niños`, ¿por qué el token de la segunda fusión no se puede mostrar como texto? ¿Cómo lo resuelve `decodificar`?

8. Explica por qué un vocabulario de 256 bytes puede representar cualquier texto. Relaciónalo con el problema de `rosa` del capítulo 2.

9. Según la figura del capítulo, ¿cuántos bytes por token se obtienen con 128 y con 512 fusiones? ¿Por qué cada fusión nueva ahorra menos que la anterior?

10. Cambia la constante `FUSIONES` (línea 6 del programa) a 64 y después a 512, y ejecútalo cada vez. Compara el número de tokens del corpus y los tokens más largos. Con 512 fusiones el programa tarda algo más y `<|fin|>` pasa a ser el token 768.

11. ¿Por qué la frase en inglés necesita más tokens que la española? ¿Qué pasaría con un tokenizador entrenado con textos en inglés?

12. ¿Por qué `<|fin|>` se añade como identificador y no como texto? ¿Qué ocurre si alguien escribe esos caracteres en un mensaje?

13. Análisis de segmentación. Elige diez palabras: cinco que aparezcan a menudo en este libro y cinco que no aparezcan nunca. Codifícalas con el tokenizador del capítulo, anota sus tokens en una tabla, comprueba la ida y vuelta de cada una y escribe cinco líneas sobre lo que observas.

## Referencias

Estructura basada en [minbpe](https://github.com/karpathy/minbpe) y en la lección [Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE), de Andrej Karpathy. Corpus, ejemplos y explicaciones propios.